# Spine XR Augmentation — Project 3 Colab Runner

Google Colab Pro+ (A100) üzerinde koşulmak üzere tasarlandı. Tüm ağır eğitim buradan yürütülür.

## Sıralama

1. Bootstrap (Drive mount, repo + dataset.rar'ı SSD'ye çek, outputs'u Drive'a sembolik bağla, pip install)
2. `01_audit` → `02_data_splitter`
3. `03_train_classifier` — 4 Case × 2 Backbone (VGG16, InceptionV3) — baseline (no aug)
4. (Sonraki milestone'larda) Phase 04 traditional, Phase 05–07 WGAN, Phase 08 hybrid, Phase 09 final report

## 1. Bootstrap

In [ ]:
# 1. Drive Mount
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
from pathlib import Path

# 2. Çalışma Alanını Yerel SSD'de Ayarla (A100'ün maksimum hızı için)
LOCAL_ROOT = Path('/content/spine-xr-augmentation-study-3')
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(LOCAL_ROOT)

# 3. Kodları Drive'dan Yerele Kopyala
DRIVE_REPO_PATH = Path('/content/drive/MyDrive/spine-xr/spine-xr-augmentation-study-3')
!cp -r {DRIVE_REPO_PATH}/* .

# 4. Dataset'i SSD'ye Çek ve Aç (Dataset Drive'da .rar olarak durmalı)
# Klasör Adı "dataset" olmalı — configs/base.yaml relatif `dataset/...` yolları kullanır.
DRIVE_DATASET_PATH = Path('/content/drive/MyDrive/spine-xr/dataset.rar')
!unrar x -o+ {DRIVE_DATASET_PATH} {LOCAL_ROOT}/

# 5. Çıktıların (Outputs) Kaybolmaması İçin Drive'a Bağla
DRIVE_OUTPUTS = Path('/content/drive/MyDrive/spine-xr/outputs-project-3')
DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)

if os.path.exists('outputs') and not os.path.islink('outputs'):
    shutil.rmtree('outputs')
elif os.path.islink('outputs'):
    os.remove('outputs')
os.symlink(DRIVE_OUTPUTS, 'outputs')

print(f"Çalışma dizini (SSD): {os.getcwd()}")
!ls -l

In [ ]:
!pip install -q -r requirements.txt
!python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

## 2. Audit + Splits

In [ ]:
!python scripts/01_audit.py --config configs/base.yaml
!python scripts/02_data_splitter.py --config configs/base.yaml --cases configs/cases.yaml
!cat outputs/01_audit/audit_report.md
!echo '---'
!cat outputs/02_splits/splits_summary.md

## 3. Baseline classifier — 4 Case × 2 Backbone

### Smoke Test

In [ ]:
# SMOKE: en küçük case (case_4) + VGG16 + 1 epoch — pipeline çalışıyor mu kontrolü
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_4 \
    --backbones-filter vgg16 \
    --epochs 1 \
    --out-tag 03_smoke

### Case 1 / VGG16

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_1 \
    --backbones-filter vgg16

### Case 1 / InceptionV3

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_1 \
    --backbones-filter inception_v3

### Case 2 / VGG16

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_2 \
    --backbones-filter vgg16

### Case 2 / InceptionV3

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_2 \
    --backbones-filter inception_v3

### Case 3 / VGG16

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_3 \
    --backbones-filter vgg16

### Case 3 / InceptionV3

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_3 \
    --backbones-filter inception_v3

### Case 4 / VGG16

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_4 \
    --backbones-filter vgg16

### Case 4 / InceptionV3

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_4 \
    --backbones-filter inception_v3

### Sonuçları Özetle (8 hücre tamamlandıktan sonra)

In [ ]:
# Tüm cell'leri tarayıp tek bir summary.md/csv üreten yardımcı betik (sonraki milestone'a kadar elle bu hücre yeter)
import json, pandas as pd
from pathlib import Path

rows = []
for case_dir in sorted(Path('outputs/03_baseline').glob('case_*')):
    for bb_dir in sorted(case_dir.glob('*')):
        m_path = bb_dir / 'metrics.json'
        if not m_path.exists():
            continue
        m = json.loads(m_path.read_text())
        row = {
            'case': m['case'], 'backbone': m['backbone'],
            'best_epoch': m['best_epoch'],
            'best_test_macro_f1': round(m['best_test_macro_f1'], 4),
            'best_val_macro_f1': round(m['best_val_macro_f1'], 4) if m['best_val_macro_f1'] == m['best_val_macro_f1'] else float('nan'),
        }
        for c, pc in m['best_test_metrics']['per_class'].items():
            row[f'F1__{c}'] = round(pc['f1'], 4)
        rows.append(row)
df = pd.DataFrame(rows)
Path('outputs/03_baseline').mkdir(parents=True, exist_ok=True)
df.to_csv('outputs/03_baseline/summary.csv', index=False)
Path('outputs/03_baseline/summary.md').write_text('# Baseline summary\n\n' + df.to_markdown(index=False))
df